# 3. LSST matching and analysis products

This notebook demonstrates the collection-aware matching API and the standard products: summaries, distributions, maps, cutouts, and PDF reports.

In [ ]:
from pathlib import Path
from target_selection import open_collection, report, target_io

collection = open_collection("DP2")
targets = target_io.load("lastberu_watchlist")
output = Path("outputs/lastberu_watchlist_dp2")
output.mkdir(parents=True, exist_ok=True)


In [ ]:
coverage = collection.get_visit_coverage(targets, cache_dir=output / "cache" / "visits", bands=tuple("ugrizy"), max_workers=2)
coverage = collection.get_coadd_coverage(coverage, cache_path=output / "cache" / "coadds.csv", bands=tuple("ugrizy"))
positions = coverage[["target_id", "ra_deg", "dec_deg"]]
properties = collection.get_property_map_values(positions, properties=("maglim", "psf_size", "exposure_time", "sky_noise"), bands=tuple("ugrizy"), cache_path=output / "cache" / "properties.csv")
coverage = coverage.merge(properties, on="target_id", how="left")


In [ ]:
report.match_summary(coverage, output, bands=tuple("ugrizy"), category_column="category", category_label="Category", name="lastberu_watchlist")
report.property_distributions(coverage, output, bands=tuple("ugrizy"), category_column="category", category_label="Category", name="lastberu_watchlist")
report.sky_maps(coverage, output, bands=tuple("ugrizy"), name="lastberu_watchlist")

In [ ]:
manifest = collection.get_image(coverage, image_type="coadd", output_dir=output / "cutouts" / "pngs", fits_dir=output / "cutouts" / "fits", bands=tuple("ugrizy"), size_arcsec=20.0, category_column="category", add_rgb=False, reuse_existing=True)
report.main(coverage, fits_dir=output / "cutouts" / "fits", asset_dir=output, output_path=output / "report.pdf", bands=tuple("ugrizy"))
